In [1]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import timm

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

torch.backends.cudnn.benchmark = True

# ============================================
# CONFIG
# ============================================

DATASET_PATH = r"I:\Research\Deepfake Detection program\dataset" 

NUM_FRAMES = 15
IMG_SIZE = 224
BATCH_SIZE = 4
EPOCHS = 50

# ============================================
# DATASET
# ============================================

class DeepfakeDataset(Dataset):

    def __init__(self, video_paths, labels):
        self.video_paths = video_paths
        self.labels = labels

    def extract_frames(self, video_path):

        cap = cv2.VideoCapture(video_path)

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        frames = []

        if total_frames <= 0:
            cap.release()
            return torch.zeros(NUM_FRAMES,3,IMG_SIZE,IMG_SIZE)

        frame_ids = np.linspace(
            0,
            max(total_frames - 1, 0),
            NUM_FRAMES,
            dtype=int
        )

        for frame_id in frame_ids:

            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)

            success, frame = cap.read()

            if not success:
                frame = np.zeros(
                    (IMG_SIZE, IMG_SIZE, 3),
                    dtype=np.uint8
                )

            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            frame = cv2.resize(
                frame,
                (IMG_SIZE, IMG_SIZE)
            )

            frame = frame.astype(np.float32) / 255.0

            frame = torch.tensor(frame).permute(2,0,1)

            frames.append(frame)

        cap.release()

        return torch.stack(frames)

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):

        frames = self.extract_frames(
            self.video_paths[idx]
        )

        label = self.labels[idx]

        return frames, label
# ============================================
# LOAD VIDEO PATHS
# ============================================

video_paths = []
labels = []

real_folder = os.path.join(DATASET_PATH, "real")
fake_folder = os.path.join(DATASET_PATH, "fake")

for file in os.listdir(real_folder):

    if file.endswith((".mp4",".avi",".mov")):

        video_paths.append(
            os.path.join(real_folder,file)
        )

        labels.append(0)

for file in os.listdir(fake_folder):

    if file.endswith((".mp4",".avi",".mov")):

        video_paths.append(
            os.path.join(fake_folder,file)
        )

        labels.append(1)

print("Total videos:", len(video_paths))

# ============================================
# TRAIN VALIDATION SPLIT
# ============================================

train_paths, val_paths, train_labels, val_labels = train_test_split(
    video_paths,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

train_dataset = DeepfakeDataset(
    train_paths,
    train_labels
)

val_dataset = DeepfakeDataset(
    val_paths,
    val_labels
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)



# ============================================
# ATTENTION BLOCK
# ============================================

class AttentionBlock(nn.Module):

    def __init__(self, feature_dim):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(feature_dim, feature_dim // 2),
            nn.ReLU(),
            nn.Linear(feature_dim // 2, 1)
        )

    def forward(self, x):

        weights = self.attention(x)

        weights = torch.softmax(
            weights,
            dim=1
        )

        attended = (
            weights * x
        ).sum(dim=1)

        return attended

# ============================================
# FFT BRANCH
# ============================================

class FFTBranch(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.fc = nn.Sequential(
            nn.Linear(input_dim,256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256,128)
        )

    def forward(self,x):

        fft = torch.fft.fft(
            x,
            dim=-1
        )

        fft = torch.abs(fft)

        fft = fft.mean(dim=1)

        return self.fc(fft)

# ============================================
# FUSION
# ============================================

class FeatureFusion(nn.Module):

    def __init__(self):
        super().__init__()

        self.fusion = nn.Sequential(
            nn.Linear(1280+128,512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512,256)
        )

    def forward(self,a,b):

        x = torch.cat([a,b],dim=1)

        return self.fusion(x)

# ============================================
# MODEL
# ============================================

class DeepfakeEfficientNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "efficientnet_b0",
            pretrained=True,
            num_classes=0
        )

        self.feature_dim = 1280

        self.attention = AttentionBlock(
            self.feature_dim
        )

        self.fft_branch = FFTBranch(
            self.feature_dim
        )

        self.fusion = FeatureFusion()

        self.classifier = nn.Sequential(
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128,2)
        )

    def forward(self,x):

        B,T,C,H,W = x.shape

        x = x.view(
            B*T,
            C,
            H,
            W
        )

        features = self.backbone(x)

        features = features.view(
            B,
            T,
            -1
        )

        attention_feature = self.attention(
            features
        )

        fft_feature = self.fft_branch(
            features
        )

        fused = self.fusion(
            attention_feature,
            fft_feature
        )

        output = self.classifier(
            fused
        )

        return output

# ============================================
# TRAINING
# ============================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = DeepfakeEfficientNet().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

for epoch in range(EPOCHS):

    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for videos, labels in train_loader:

        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(videos)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        preds = outputs.argmax(1)

        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss / len(train_loader)
    train_acc = 100 * train_correct / train_total

    # =========================
    # VALIDATION
    # =========================
    model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for videos, labels in val_loader:

            videos = videos.to(device)
            labels = labels.to(device)

            outputs = model(videos)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            preds = outputs.argmax(1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss / len(val_loader)
    val_acc = 100 * val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%"
    )

print("Training Complete")

i:\Research\Deepfake Detection program\deepfake\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA Available: True
GPU: NVIDIA GeForce RTX 5060 Ti
Total videos: 2000


Epoch [1/50] | Train Loss: 0.7015 | Train Acc: 51.62% | Val Loss: 0.6944 | Val Acc: 50.25%
Epoch [2/50] | Train Loss: 0.6999 | Train Acc: 50.25% | Val Loss: 0.6900 | Val Acc: 50.50%
Epoch [3/50] | Train Loss: 0.6262 | Train Acc: 65.38% | Val Loss: 0.5250 | Val Acc: 76.50%
Epoch [4/50] | Train Loss: 0.4305 | Train Acc: 82.25% | Val Loss: 0.4191 | Val Acc: 80.25%
Epoch [5/50] | Train Loss: 0.3337 | Train Acc: 87.19% | Val Loss: 0.4340 | Val Acc: 80.50%
Epoch [6/50] | Train Loss: 0.2721 | Train Acc: 90.00% | Val Loss: 0.3111 | Val Acc: 87.00%
Epoch [7/50] | Train Loss: 0.2004 | Train Acc: 92.06% | Val Loss: 0.3176 | Val Acc: 86.75%
Epoch [8/50] | Train Loss: 0.1622 | Train Acc: 93.81% | Val Loss: 0.3985 | Val Acc: 85.00%
Epoch [9/50] | Train Loss: 0.1759 | Train Acc: 93.81% | Val Loss: 0.4255 | Val Acc: 82.25%
Epoch [10/50] | Train Loss: 0.1253 | Train Acc: 95.44% | Val Loss: 0.4840 | Val Acc: 83.00%
Epoch [11/50] | Train Loss: 0.1153 | Train Acc: 96.12% | Val Loss: 0.3310 | Val Acc: 88.0